In [1]:
from io import StringIO

import numpy as np
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC
from scipy.interpolate import interp1d
from sora import Body, EphemPlanete, Observer
from sora.prediction import prediction

SORA version: 0.3.3


In [2]:
# Hiperparámetros
rock = 'Silesia'
obs = 'W63'
epoch_jpl = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1m'
}
epoch_mpc = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1min'
}
mag_lim = 16
sora_step = 10 # Paso en segundos del muestreo de predicción de SORA

In [3]:
# Bajamos efemérides del asteroide con horizons
body_jpl = Horizons(id=rock, epochs=epoch_jpl, location=obs)
eph_jpl = body_jpl.ephemerides()
data_jpl = eph_jpl['datetime_jd', 'RA', 'DEC', 'delta'] # Datos que pide SORA en objeto efemérides
sigma_jpl = eph_jpl['RA_3sigma', 'DEC_3sigma'] # incertidumbre 3sigma para la ascención recta y declinación en ese instante
error_jpl = eph_jpl['SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma'] # característica del elipse de error (matriz de covarianza diagonalizada)

# Bajamos efemérides del asteroide con MPCES
body_mpc = MPC.query_object('asteroid', name=rock)
eph_mpc = MPC.get_ephemeris(
  rock, 
  step=epoch_mpc['step'], 
  start=epoch_mpc['start'], 
  number=1441, 
  location=obs
)
eph_mpc['Date_jd'] = Time(eph_mpc['Date']).jd # Las fechas tienen que estar en formato juliano
data_mpc = eph_mpc['Date_jd', 'RA', 'Dec', 'Delta'] #  Datos que pide SORA en objeto efemérides
error_mpc = eph_mpc['Uncertainty 3sig', 'Unc. P.A.'] # características del error principal (componente principal)

In [39]:
eph_mpc['Date'].__dict__

{'_time': <astropy.time.formats.TimeISO at 0x14b93683d40>,
 '_location': None,
 '_format': 'iso',
 'SCALES': ('tai', 'tcb', 'tcg', 'tdb', 'tt', 'ut1', 'utc'),
 'info': name = Date
 dtype = object
 class = Time
 n_bad = 0
 length = 1441,
 'cache': defaultdict(dict,
             {'mask': array([False, False, False, ..., False, False, False], shape=(1441,))})}

In [6]:
# Función para convertir tabla astropy de query a ephem planete de sora (ya permite predicciones con cualquier query)
def ephem_sora(name, data):
  df = data.to_pandas()
  buffer = StringIO()
  df.to_csv(buffer, sep=' ', header=False, index=False, float_format='%.12f')
  buffer.seek(0)
  return EphemPlanete(ephem=buffer, name=name)

# Instanciamos objeto de efemérides de JPL y MPC
eph_jpl_sora = ephem_sora(rock, data_jpl)
eph_mpc_sora = ephem_sora(rock, data_mpc)

# Variable controlada (objeto)
sora_body = Body(rock)

# Instanciamos objeto menor con SORA utilizando los datos de jpl y mpc
sora_body_jpl = Body(rock, ephem=eph_jpl_sora)
sora_body_mpc = Body(rock, ephem=eph_mpc_sora)

# Instanciamos observador
sora_obs = Observer(name=obs, code=obs)
jd = np.array(data_jpl['datetime_jd'])
ra = np.array(data_jpl['RA'])
dec = np.array(data_jpl['DEC'])
delta = np.array(data_jpl['delta'])


# Monkey patch
ra_interp = interp1d(jd, ra)
dec_interp = interp1d(jd, dec)
dist_interp = interp1d(jd, delta)

def get_position(time, observer='geocenter'):
    jd_query = np.atleast_1d(time.jd)

    return SkyCoord(
        ra_interp(jd_query)*u.deg,
        dec_interp(jd_query)*u.deg,
        distance=dist_interp(jd_query)*u.au
    )

eph_jpl_sora.get_position = get_position

Obtaining data for Silesia from SBDB
Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(


Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(


In [ ]:
# Controlado
sora_pred = prediction(
  body=sora_body,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

1 occultations found.


In [ ]:
# Experimento
jpl_pred = prediction(
  body=sora_body_jpl,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

No stellar occultation was found.


In [24]:
import inspect

from sora.ephem import EphemHorizons, EphemPlanete

print(inspect.getsource(EphemHorizons))
print('\n')
print(inspect.getsource(EphemPlanete))

class EphemHorizons(BaseEphem):
    """Obtains the ephemeris from Horizons/JPL service.

    Note
    ----
    Web tool URL: https://ssd.jpl.nasa.gov/horizons.cgi


    Attributes
    ----------
    name : `str`, required
        Name of the object to search in the JPL database.

    id_type: `str`, default='smallbody'
        Type of object options: ``smallbody``, ``majorbody`` (planets but
        also anything that is not a small body), ``designation``, ``name``,
        ``asteroid_name``, ``comet_name``, ``id`` (Horizons id number), or
        ``smallbody`` (find the closest match under any id_type).

    radius : `int`, `float`, default: online database
        Object radius, in km.

    error_ra : `int`, `float`, default: online database
        Ephemeris RA*cosDEC error, in arcsec.

    error_dec : `int`, `float`, default: online database
        Ephemeris DEC error, in arcsec.

    mass : `int`, `float`, default=0
        Object mass, in kg.

    H : `int`, `float`, default=NaN